# FSMA Bronze & Silver Ingestion

Reads table configuration from `fsma_table_config.json` and processes each M3 source table through:
- **Bronze**: Raw append with audit columns (`from_timestamp`, `to_timestamp`, `current_flag`, `loadtimestamp`)
- **Silver**: SCD Type 2 merge using configured primary keys and LMTS watermark

**Load Modes:**
- **Initial**: Full extract when bronze table does not exist or is empty
- **Incremental**: Pulls records where `LMTS >= (max existing LMTS - 24 hours)`

In [0]:
spark.conf.set(
    "fs.azure.account.key.stscmddidev.dfs.core.windows.net",
    dbutils.secrets.get(
        scope="mddi-kv-scope",
        key="CONTAINER-STSCMDDIDEV-KEY"
    )
)

print("ADLS storage key configured.")
#print("ADLS storage key configured.")

In [0]:
import json
from pyspark.sql.functions import col, lit, current_timestamp, max as spark_max, to_date
from delta.tables import DeltaTable
from datetime import datetime, timedelta

# ── Read table configuration ─────────────────────────────────────
config_path = "/Workspace/Users/rajender.bekkary@kroger.com/fsma_tbl_config.json"

with open(config_path, "r") as f:
    CONFIG = json.load(f)

# ============================================================
# Detect Environment from ADLS Base Location
# DEV  -> CONO = 250
# PRD  -> CONO = 100
# ============================================================

bronze_base_location = CONFIG["layers"]["bronze"]["base_location"].lower()

if "dev" in bronze_base_location:
    CURRENT_ENV = "dev"
elif "prod" in bronze_base_location:
    CURRENT_ENV = "prd"
else:
    raise Exception(
        f"Unable to determine environment from bronze base location: {bronze_base_location}"
    )

COMPANY_FILTER = CONFIG["company_filter"][CURRENT_ENV]

print("=" * 80)
print(f"Config Path          : {config_path}")
print(f"Bronze Base Location : {bronze_base_location}")
print(f"Detected Environment : {CURRENT_ENV.upper()}")
print(f"CONO Filter          : {COMPANY_FILTER}")
print("=" * 80)

# ============================================================
# Read remaining configuration
# ============================================================

SOURCE_SCHEMA = CONFIG["source"]["schema"]
TABLE_CONFIG = CONFIG["tables"]

BRONZE_SCHEMA = CONFIG["layers"]["bronze"]["target_schema"]
SILVER_SCHEMA = CONFIG["layers"]["silver"]["target_schema"]

BRONZE_BASE_LOCATION = CONFIG["layers"]["bronze"]["base_location"]
SILVER_BASE_LOCATION = CONFIG["layers"]["silver"]["base_location"]

print(f"Loaded configuration for {len(TABLE_CONFIG)} tables:")

for t in TABLE_CONFIG:
    print(
        f"  {SOURCE_SCHEMA}.{t['source_table']:10s} "
        f"-> {BRONZE_SCHEMA}.bronze_{t['target_name']} / "
        f"{SILVER_SCHEMA}.silver_{t['target_name']}"
    )

# ── JDBC connection setup ────────────────────────────────────────
client_id = dbutils.secrets.get(scope="mddi-kv-scope", key="client-id")
client_secret = dbutils.secrets.get(scope="mddi-kv-scope", key="client-secret")

JDBC_URL = (
    "jdbc:sqlserver://sc-mfg-datastore-stagea21.database.windows.net:1433;"
    "database=sc-mfg-datastoredb-stagea21;"
    "encrypt=true;"
    "trustServerCertificate=true;"
    "authentication=ActiveDirectoryServicePrincipal"
)

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {BRONZE_SCHEMA}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SILVER_SCHEMA}")

print(f"\nJDBC connection configured.")
print(f"Bronze schema : {BRONZE_SCHEMA}")
print(f"Silver schema : {SILVER_SCHEMA}")

run_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
print(f"Process start : {run_time}")

In [0]:
def process_table(table_cfg):
    """
    Bronze + Silver ingestion for one FSMA source table.

    Environment/company filtering:
      - DEV  -> CONO = 250
      - PRD  -> CONO = 100
      - COMPANY_FILTER is calculated in the config/setup cell.
      - Every source extract query applies CONO = COMPANY_FILTER.

    Incremental logic:
      - No fixed 24-hour or 72-hour lookback.
      - Resumes dynamically from the last successful Bronze watermark.
      - LMTS uses > max(LMTS)
      - LMDT uses >= max(LMDT)

    Silver logic:
      - Simple overwrite from Bronze ALWAYS — even if no new source records
      - Deduplicated on PK columns — latest record per PK based on watermark
      - No SCD2, no current_flag, no history
    """

    source_table_raw = table_cfg["source_table"]
    target_name      = table_cfg["target_name"]
    primary_keys     = table_cfg["primary_keys"]

    source_table = f"{SOURCE_SCHEMA}.{source_table_raw}"
    bronze_table = f"{BRONZE_SCHEMA}.bronze_{target_name}"
    silver_table = f"{SILVER_SCHEMA}.silver_{target_name}"

    bronze_location = f"{BRONZE_BASE_LOCATION}/bronze_{target_name}"
    silver_location = f"{SILVER_BASE_LOCATION}/silver_{target_name}"

    incoming_df    = None
    incoming_count = 0
    silver_count   = 0
    is_incremental = False
    watermark_col  = "LMTS"

    print(f"\n{'=' * 80}")
    print(f"Processing source : {source_table}")
    print(f"CONO filter       : {COMPANY_FILTER}")
    print(f"Bronze target     : {bronze_table}")
    print(f"Bronze location   : {bronze_location}")
    print(f"Silver target     : {silver_table}")
    print(f"Silver location   : {silver_location}")
    print(f"{'=' * 80}")

    try:
        # ── Determine load mode and watermark column ──────────────────────────
        if spark.catalog.tableExists(bronze_table):
            bronze_existing_df = spark.table(bronze_table)
            bronze_cols        = bronze_existing_df.columns

            if "LMTS" in bronze_cols:
                watermark_col = "LMTS"
            elif "LMDT" in bronze_cols:
                watermark_col = "LMDT"
            else:
                raise ValueError(
                    f"Bronze table exists but neither LMTS nor LMDT found: {bronze_table}"
                )

            existing_count = bronze_existing_df.limit(1).count()

            if existing_count > 0:
                is_incremental = True
                print("Mode: INCREMENTAL")
            else:
                print("Mode: INITIAL - Bronze table exists but empty")
        else:
            print("Mode: INITIAL - Bronze table does not exist")

        # ── Build source query ────────────────────────────────────────────────
        if is_incremental:
            max_wm = (
                spark.table(bronze_table)
                .select(spark_max(watermark_col))
                .collect()[0][0]
            )

            if max_wm is not None:
                cutoff = int(max_wm)

                if watermark_col == "LMTS":
                    cutoff_display = datetime.fromtimestamp(cutoff / 1000)
                    query = f"""
                        SELECT * FROM {source_table}
                        WHERE CONO = {COMPANY_FILTER}
                          AND {watermark_col} > {cutoff}
                    """
                    print("Dynamic incremental logic: LMTS")
                    print(f"Last Bronze watermark : {watermark_col} = {cutoff} ({cutoff_display})")
                    print(f"Source filter         : CONO = {COMPANY_FILTER} AND {watermark_col} > {cutoff}")

                else:
                    query = f"""
                        SELECT * FROM {source_table}
                        WHERE CONO = {COMPANY_FILTER}
                          AND {watermark_col} >= {cutoff}
                    """
                    print("Dynamic incremental logic: LMDT")
                    print(f"Last Bronze watermark : {watermark_col} = {cutoff}")
                    print(f"Source filter         : CONO = {COMPANY_FILTER} AND {watermark_col} >= {cutoff}")

            else:
                print(f"Warning: No valid {watermark_col} - falling back to full extract")
                query = f"SELECT * FROM {source_table} WHERE CONO = {COMPANY_FILTER}"

        else:
            query = f"SELECT * FROM {source_table} WHERE CONO = {COMPANY_FILTER}"
            print("Initial load query: full extract with CONO filter")
            print(f"Source filter      : CONO = {COMPANY_FILTER}")

        # ── Read from Azure SQL source ────────────────────────────────────────
        incoming_df = (
            spark.read.format("jdbc")
            .option("url",      JDBC_URL)
            .option("query",    query)
            .option("user",     client_id)
            .option("password", client_secret)
            .option("driver",   "com.microsoft.sqlserver.jdbc.SQLServerDriver")
            .load()
        )

        incoming_df    = incoming_df.cache()
        incoming_count = incoming_df.count()

        print(f"Records fetched from source: {incoming_count}")

        # ── Detect watermark for initial load ─────────────────────────────────
        if not is_incremental:
            if "LMTS" in incoming_df.columns:
                watermark_col = "LMTS"
            elif "LMDT" in incoming_df.columns:
                watermark_col = "LMDT"
            else:
                raise ValueError(
                    f"Source table has neither LMTS nor LMDT: {source_table}"
                )

        print(f"Using watermark column: {watermark_col}")

        # ═════════════════════════════════════════════════════════════════════
        # BRONZE - Append only if new records fetched from source
        # ═════════════════════════════════════════════════════════════════════
        if incoming_count > 0:
            if watermark_col == "LMTS":
                from_ts_expr = (col("LMTS") / 1000).cast("timestamp")
            else:
                from_ts_expr = to_date(col("LMDT").cast("string"), "yyyyMMdd").cast("timestamp")

            bronze_df = (
                incoming_df
                .withColumn("from_timestamp", from_ts_expr)
                .withColumn("loadtimestamp",  current_timestamp())
            )

            bronze_df.write.format("delta") \
                .mode("append") \
                .option("path", bronze_location) \
                .saveAsTable(bronze_table)

            print(f"Bronze: Appended {incoming_count} records to {bronze_table}")

        else:
            print(f"No new records from source — Bronze append skipped")

        # ═════════════════════════════════════════════════════════════════════
        # SILVER - Always refresh from full Bronze
        # Deduplicated on PK columns — latest record per PK
        # ═════════════════════════════════════════════════════════════════════
        from pyspark.sql.window import Window
        from pyspark.sql.functions import row_number, desc

        if not spark.catalog.tableExists(bronze_table):
            print(f"Bronze table does not exist yet — Silver refresh skipped")
        else:
            bronze_full_df = spark.table(bronze_table)

            # ── Deduplicate — keep latest record per PK ───────────────────────
            window_spec = Window.partitionBy(primary_keys).orderBy(desc(watermark_col))

            silver_df = bronze_full_df \
                .withColumn("_rn", row_number().over(window_spec)) \
                .filter(col("_rn") == 1) \
                .drop("_rn") \
                .withColumn("silver_loadtimestamp", current_timestamp())

            silver_count = silver_df.count()

            silver_df.write.format("delta") \
                .mode("overwrite") \
                .option("overwriteSchema", "true") \
                .option("path", silver_location) \
                .saveAsTable(silver_table)

            print(f"Silver: Overwritten with {silver_count} deduplicated records")
            print(f"Silver: Latest snapshot written to {silver_table}")

    except Exception as e:
        error_msg = str(e)
        if any(phrase in error_msg.lower() for phrase in [
            "invalid object name",
            "table not found",
            "does not exist",
            "could not be found"
        ]):
            print(f"WARNING: Table not found in source: {source_table} - skipping")
        else:
            print(f"ERROR processing {source_table}: {error_msg}")
            raise

    finally:
        if incoming_df is not None:
            incoming_df.unpersist()

    return {
        "source_table":    source_table,
        "target_name":     target_name,
        "load_mode":       "INCREMENTAL" if is_incremental else "INITIAL",
        "watermark_col":   watermark_col,
        "records_fetched": incoming_count,
        "bronze_appended": incoming_count,
        "silver_count":    silver_count,
        "status":          "SUCCESS",
        "error_message":   None
    }

In [0]:
# Incremental test run for smallest table only: CMNDIV

#test_cfg = next(t for t in TABLE_CONFIG if t["source_table"] == "CMNDIV")

print("Running incremental test for smallest table: only for testing incremental flow")
#print(test_cfg)

#process_table(test_cfg)

In [0]:
# ── Process all tables and capture run metadata ───────────────────────────────
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType

print(f"Starting ingestion for {len(TABLE_CONFIG)} tables...")

job_start_time = datetime.now()
print(f"Timestamp: {job_start_time}\n")

run_metadata = []

for table_cfg in TABLE_CONFIG:
    table_start = datetime.now()
    meta = process_table(table_cfg)
    table_end = datetime.now()

    if meta:
        meta["table_start_time"] = table_start
        meta["table_end_time"]   = table_end
        meta["job_start_time"]   = job_start_time
        run_metadata.append(meta)

print(f"\n{'='*60}")
print("All tables processed.")
print(f"{'='*60}")
job_end_time = datetime.now()
print(f"Job End Time: {job_end_time}")

# ── Single write to metadata table ───────────────────────────────────────────
schema = StructType([
    StructField("source_table",    StringType(),    True),
    StructField("target_name",     StringType(),    True),
    StructField("load_mode",       StringType(),    True),
    StructField("watermark_col",   StringType(),    True),
    StructField("records_fetched", IntegerType(),   True),
    StructField("bronze_appended", IntegerType(),   True),
    StructField("silver_count",    IntegerType(),   True),
    StructField("status",          StringType(),    True),
    StructField("error_message",   StringType(),    True),
    StructField("table_start_time",TimestampType(), True),
    StructField("table_end_time",  TimestampType(), True),
    StructField("job_start_time",  TimestampType(), True)
])

metadata_table = f"{BRONZE_SCHEMA}.pipeline_run_log"

spark.createDataFrame(run_metadata, schema=schema) \
    .write.format("delta") \
    .mode("append") \
    .saveAsTable(metadata_table)

print(f"\nRun metadata saved to : {metadata_table}")
display(spark.table(metadata_table).orderBy("job_start_time", "table_start_time"))

In [0]:
print("Bronze Tables")
display(
    spark.sql("""
        SHOW TABLES IN hive_metastore.bulk_receiving_bronze_dev
    """).filter("tableName <> 'bronze_stock_transactions1'")
)

print("Silver Tables")
display(
    spark.sql("""
        SHOW TABLES IN hive_metastore.bulk_receiving_silver_dev
    """).filter("tableName <> 'silver_stock_transactions1'")
)

In [0]:
for t in TABLE_CONFIG:
    table_name = f"hive_metastore.bulk_receiving_bronze_dev.bronze_{t['target_name']}"

    try:
        cnt = spark.sql(
            f"SELECT COUNT(*) cnt FROM {table_name}"
        ).collect()[0]["cnt"]

        print(f"{table_name:<100} {cnt:,}")

    except Exception as e:
        print(f"{table_name} : ERROR - {e}")

In [0]:
# ══════════════════════════════════════════════════════════════════════════════
# Pipeline Run Log — Metadata Verification
# 
# Intent  : Validate that pipeline_run_log table has correctly captured
#           one metadata row per table per job run with accurate counts,
#           timestamps and status fields.
#
# Checks  :
#   Check 1 — Row count per job run
#   Check 2 — Full detail of latest run
#   Check 3 — Any failed tables
#   Check 4 — Null check on key fields
#
# Table   : hive_metastore.bulk_receiving_bronze_dev.pipeline_run_log
# ══════════════════════════════════════════════════════════════════════════════

from pyspark.sql.functions import col

metadata_table = f"{BRONZE_SCHEMA}.pipeline_run_log"

# ── Check 1: Row count per run ────────────────────────────────────────────────
print("Check 1: Rows per job run")
spark.table(metadata_table) \
    .groupBy("job_start_time") \
    .count() \
    .orderBy("job_start_time", ascending=False) \
    .show(truncate=False)

# ── Check 2: Full detail of latest run ───────────────────────────────────────
print("Check 2: Latest run detail")
latest_run = (
    spark.table(metadata_table)
    .orderBy("job_start_time", ascending=False)
    .limit(1)
    .collect()[0]["job_start_time"]
)

display(
    spark.table(metadata_table)
    .filter(col("job_start_time") == latest_run)
    .select(
        "source_table",
        "load_mode",
        "watermark_col",
        "records_fetched",
        "bronze_appended",
        "silver_count",
        "status",
        "error_message",
        "table_start_time",
        "table_end_time"
    )
    .orderBy("table_start_time")
)

# ── Check 3: Any failures ─────────────────────────────────────────────────────
print("Check 3: Failed tables")
failed = spark.table(metadata_table).filter("status = 'FAILED'")
failed_count = failed.count()
if failed_count == 0:
    print("✅ No failures found")
else:
    print(f"❌ {failed_count} failures found")
    display(failed)

# ── Check 4: Null check on key fields ────────────────────────────────────────
print("Check 4: Null check on key fields")
nulls = spark.table(metadata_table).filter(
    col("source_table").isNull()  |
    col("load_mode").isNull()     |
    col("status").isNull()        |
    col("job_start_time").isNull()
)
null_count = nulls.count()
if null_count == 0:
    print("✅ No nulls found in key fields")
else:
    print(f"❌ {null_count} rows with nulls found")
    display(nulls)